# Install necessary Libraries

In [ ]:
!pip install pycocotools opencv-python matplotlib timm

In [ ]:
import os
import shutil
import random

# ===== SOURCE (READ-ONLY) =====
SRC_BASE = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train"
SRC_IMG = os.path.join(SRC_BASE, "image")
SRC_ANNO = os.path.join(SRC_BASE, "annos")

# ===== DESTINATION (WRITEABLE) =====
DEST_BASE = "/kaggle/working/train_subset"
DEST_IMG = os.path.join(DEST_BASE, "image")
DEST_ANNO = os.path.join(DEST_BASE, "annos")

# ===== CREATE OUTPUT DIR =====
os.makedirs(DEST_IMG, exist_ok=True)
os.makedirs(DEST_ANNO, exist_ok=True)

# ===== GET ALL IMAGES =====
all_images = sorted([f for f in os.listdir(SRC_IMG) if f.endswith(".jpg")])

# ===== SELECT HOW MANY YOU WANT =====
N = 100   # 🔥 change this (or use ALL)

# Option 1: subset
selected_images = random.sample(all_images, min(N, len(all_images)))

# Option 2 (uncomment to take ALL)
# selected_images = all_images

# ===== COPY =====
copied = 0
missing_annos = 0

for img_file in selected_images:
    base = os.path.splitext(img_file)[0]
    anno_file = base + ".json"

    img_src = os.path.join(SRC_IMG, img_file)
    anno_src = os.path.join(SRC_ANNO, anno_file)

    if os.path.exists(anno_src):
        shutil.copy(img_src, os.path.join(DEST_IMG, img_file))
        shutil.copy(anno_src, os.path.join(DEST_ANNO, anno_file))
        copied += 1
    else:
        missing_annos += 1

print(f"✅ Copied: {copied}")
print(f"⚠️ Missing annotations: {missing_annos}")
print(f"📁 Output folder: {DEST_BASE}")

In [ ]:
import shutil

# Folder to zip
folder_path = "/kaggle/working/train_subset"

# Output zip file
zip_path = "/kaggle/working/train_subset.zip"

# Create zip
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

print(f"✅ Zipped at: {zip_path}")

# 1. Task 1: Multi-label Classification

## Dataset extraction

Above our goal is to convert image + JSOn annotations to proper clean classification dataset like:

- dataset/
  
      - train/
          - images/
          - labels.csv
  
      - val/
          - images/
          - labels.csv

We expect labels.csv to look like

image_name,coat,dress,vect,skirt,...
000001.jpg,0,1,0,0
000002.jpg,1,0,1,9

For Task1 we need:

- train/image
- train/annos
- validation/image
- validation/anno
  

### 1.1 Extract Categories from the dataset

In [ ]:
import os
import json
from tqdm import tqdm

def extract_categories(anno_dir):

    categories = set()

    files = os.listdir(anno_dir)

    for file in tqdm(files):

        if not file.endswith(".json"):
            continue

        path = os.path.join(anno_dir, file)

        with open(path) as f:
            data = json.load(f)

        for key in data:

            if "item" not in key:
                continue

            item = data[key]

            if "category_name" in item:
                categories.add(item["category_name"])

    return sorted(list(categories))

In [ ]:
train_anno_dir = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/annos"

CATEGORIES = extract_categories(train_anno_dir)

print("Number of categories:", len(CATEGORIES))
print("Categories found:\n")

for c in CATEGORIES:
    print(c)

In [ ]:
val_anno_dir = "./validation/validation/validation/annos"

val_categories = extract_categories(val_anno_dir)

print("Validation categories:\n")

for c in val_categories:
    print(c)

### 1.2 Extracting Categories freq

In [ ]:
os.chdir('/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd')

In [ ]:
import os
import json
from collections import Counter
from tqdm import tqdm

train_anno_dir = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/annos"

category_counter = Counter()

files = os.listdir(train_anno_dir)

for file in tqdm(files):

    if not file.endswith(".json"):
        continue

    path = os.path.join(train_anno_dir, file)

    with open(path) as f:
        data = json.load(f)

    for key in data:

        if "item" not in key:
            continue

        category = data[key]["category_name"]

        category_counter[category] += 1

In [ ]:
print("Category frequencies:\n")

for cat, count in category_counter.most_common():
    print(cat, ":", count)

In [ ]:
TOP5 = [cat for cat, _ in category_counter.most_common(5)]

print("Top 5 categories:")
print(TOP5)

In [ ]:
label_map = {cat:i for i,cat in enumerate(TOP5)}

print("Label mapping:")

for k,v in label_map.items():
    print(k,"->",v)

### 1.3 Generate Multi-label csv

In [ ]:
import pandas as pd

def build_labels(image_dir, anno_dir, output_csv):

    rows = []

    images = sorted(os.listdir(image_dir))

    for img in tqdm(images):

        anno_file = os.path.join(
            anno_dir,
            img.replace(".jpg",".json")
        )

        labels = [0]*5

        if os.path.exists(anno_file):

            with open(anno_file) as f:
                data = json.load(f)

            for key in data:

                if "item" not in key:
                    continue

                cat = data[key]["category_name"]

                if cat in label_map:
                    labels[label_map[cat]] = 1

        rows.append([img] + labels)

    columns = ["image"] + TOP5

    df = pd.DataFrame(rows, columns=columns)

    df.to_csv(output_csv, index=False)

    print("Saved:", output_csv)

#### 1.3.1 Create Train Labels

In [ ]:
build_labels(
    "./train/train/train/image",
    "./train/train/train/annos",
    "./train_labels_top5_new.csv"
)

#### 1.3.2 Create Validation Labels

In [ ]:
build_labels(
    "./validation/validation/validation/image",
    "./validation/validation/validation/annos",
    "./val_labels_top5_new.csv"
)

#### 1.3.3 Inspect Labels

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/train_labels_top5_new.csv")
df.head()

### 1.4 Training

We use validation dataset for the following:
- model evaluation
- early stopping
- hyperparameter tuning

In [ ]:
import torch

# Enable cuDNN auto-tuner for faster convolutions
torch.backends.cudnn.benchmark = True

#### 1.4.1 Imports

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import timm
import torch.nn as nn
from tqdm import tqdm

#### 1.4.2 Load Label CSV

In [ ]:
train_df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/train_labels_top5_new.csv")
val_df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/val_labels_top5_new.csv")

#### 1.4.3 Dataset class and Image transforms

In [ ]:
class FashionDataset(Dataset):

    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["image"])

        # ---- SAFEGUARD AGAINST CORRUPTED IMAGES ----
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            print("Skipping corrupted image:", img_path)
            return self.__getitem__((idx+1) % len(self.df))

        labels = torch.tensor(row[1:].values.astype("float32"))

        if self.transform:
            image = self.transform(image)

        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor()
])

train_dataset = FashionDataset(
    train_df,
    "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/image",
    train_transform
)

val_dataset = FashionDataset(
    val_df,
    "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/image",
    val_transform
)

#### 1.4.4 Dataset Loaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

#### 1.4.12 Training (From Scratch: Efficient Net b0)

In [ ]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

model_efficientnet_b0_scratch = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=5
)

# use both GPUs if available
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model_efficientnet_b0_scratch = torch.nn.DataParallel(model_efficientnet_b0_scratch)

model_efficientnet_b0_scratch = model_efficientnet_b0_scratch.to(device)

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_efficientnet_b0_scratch.parameters(),
    lr=3e-4
)

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_efficientnet_b0_scratch.parameters(),
    lr=3e-4
)

In [ ]:
#  Training loop

EPOCHS = 5
best_val = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_epoch(model_efficientnet_b0_scratch, train_loader)
    val_loss = validate(model_efficientnet_b0_scratch, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # overwrite latest checkpoint
    torch.save(
        model_efficientnet_b0_scratch.state_dict(),
        "/kaggle/working/latest_checkpoint_scratch_efficient_net_b0.pth"
    )

    # save best model
    if val_loss < best_val:
        best_val = val_loss

        torch.save(
            model_efficientnet_b0_scratch.state_dict(),
            "/kaggle/working/best_model_efficientnet_b0_scratch.pth"
        )

In [ ]:
# Metrics

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve

# Load best model
model_efficientnet_b0_scratch.load_state_dict(
    torch.load("/kaggle/working/best_model_efficientnet_b0_scratch.pth")
)

model_efficientnet_b0_scratch.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_efficientnet_b0_scratch(images)

        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_probs = np.vstack(all_probs)
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)


# ------------------------
# Metrics
# ------------------------

precision_micro = precision_score(all_labels, all_preds, average="micro", zero_division=0)
recall_micro = recall_score(all_labels, all_preds, average="micro", zero_division=0)
f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)

precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

roc_auc = roc_auc_score(all_labels, all_probs, average="macro")


print("\nFinal Evaluation Metrics")
print("----------------------------")
print(f"Validation Loss : {best_val:.4f}")

print("\nMicro Metrics")
print(f"Precision : {precision_micro:.4f}")
print(f"Recall    : {recall_micro:.4f}")
print(f"F1 Score  : {f1_micro:.4f}")

print("\nMacro Metrics")
print(f"Precision : {precision_macro:.4f}")
print(f"Recall    : {recall_macro:.4f}")
print(f"F1 Score  : {f1_macro:.4f}")

print("\nROC-AUC Score :", roc_auc)


# ------------------------
# ROC Curve
# ------------------------

plt.figure(figsize=(8,6))

for i in range(all_labels.shape[1]):

    fpr, tpr, _ = roc_curve(all_labels[:, i], all_probs[:, i])

    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Multi-label)")
plt.legend()

plt.show()

#### 1.4.13 Training (From Scratch: ResNet-50)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_resnet50_scratch = timm.create_model(
    "resnet50",
    pretrained=False,
    num_classes=5
)

# use multiple GPUs if available
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model_resnet50_scratch = torch.nn.DataParallel(model_resnet50_scratch)

model_resnet50_scratch = model_resnet50_scratch.to(device)

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_resnet50_scratch.parameters(),
    lr=3e-4
)

In [ ]:
EPOCHS = 5
best_val = float("inf")

final_val_loss = None

for epoch in range(EPOCHS):

    train_loss = train_epoch(model_resnet50_scratch, train_loader)
    val_loss = validate(model_resnet50_scratch, val_loader)

    final_val_loss = val_loss

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    torch.save(
        model_resnet50_scratch.state_dict(),
        "/kaggle/working/latest_checkpoint_resnet50_scratch.pth"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            model_resnet50_scratch.state_dict(),
            "/kaggle/working/best_model_resnet50_scratch.pth"
        )

print("\nFinal ResNet50 Scratch Validation Loss:", final_val_loss)

In [ ]:
# Metrics

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve

# Load best ResNet50 Scratch model
model_resnet50_scratch.load_state_dict(
    torch.load("/kaggle/working/best_model_resnet50_scratch.pth")
)

model_resnet50_scratch.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_resnet50_scratch(images)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_probs = np.vstack(all_probs)
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)


# ------------------------
# Metrics
# ------------------------

precision_micro = precision_score(all_labels, all_preds, average="micro", zero_division=0)
recall_micro = recall_score(all_labels, all_preds, average="micro", zero_division=0)
f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)

precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

roc_auc = roc_auc_score(all_labels, all_probs, average="macro")


print("\nFinal Evaluation Metrics (ResNet50 Scratch)")
print("---------------------------------------------")
print(f"Validation Loss : {best_val:.4f}")

print("\nMicro Metrics")
print(f"Precision : {precision_micro:.4f}")
print(f"Recall    : {recall_micro:.4f}")
print(f"F1 Score  : {f1_micro:.4f}")

print("\nMacro Metrics")
print(f"Precision : {precision_macro:.4f}")
print(f"Recall    : {recall_macro:.4f}")
print(f"F1 Score  : {f1_macro:.4f}")

print("\nROC-AUC Score :", roc_auc)


# ------------------------
# ROC Curve
# ------------------------

plt.figure(figsize=(8,6))

for i in range(all_labels.shape[1]):

    fpr, tpr, _ = roc_curve(all_labels[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (ResNet50 Scratch Multi-label)")
plt.legend()

plt.show()

#### 1.4.14 Training (From Scratch: MobileNet V3)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_mobilenetv3_scratch = timm.create_model(
    "mobilenetv3_large_100",
    pretrained=False,
    num_classes=5
)

# use multiple GPUs if available
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model_mobilenetv3_scratch = torch.nn.DataParallel(model_mobilenetv3_scratch)

model_mobilenetv3_scratch = model_mobilenetv3_scratch.to(device)

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_mobilenetv3_scratch.parameters(),
    lr=3e-4
)

In [ ]:
EPOCHS = 5
best_val = float("inf")

final_val_loss = None

for epoch in range(EPOCHS):

    train_loss = train_epoch(model_mobilenetv3_scratch, train_loader)
    val_loss = validate(model_mobilenetv3_scratch, val_loader)

    final_val_loss = val_loss

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # overwrite latest checkpoint
    torch.save(
        model_mobilenetv3_scratch.state_dict(),
        "/kaggle/working/latest_checkpoint_mobilenetv3_scratch.pth"
    )

    # save best model
    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            model_mobilenetv3_scratch.state_dict(),
            "/kaggle/working/best_model_mobilenetv3_scratch.pth"
        )

print("\nFinal MobileNetV3 Scratch Validation Loss:", final_val_loss)

In [ ]:
# Metrics

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve

# Load best MobileNetV3 Scratch model
model_mobilenetv3_scratch.load_state_dict(
    torch.load("/kaggle/working/best_model_mobilenetv3_scratch.pth")
)

model_mobilenetv3_scratch.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_mobilenetv3_scratch(images)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_probs = np.vstack(all_probs)
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)


# ------------------------
# Metrics
# ------------------------

precision_micro = precision_score(all_labels, all_preds, average="micro", zero_division=0)
recall_micro = recall_score(all_labels, all_preds, average="micro", zero_division=0)
f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)

precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

roc_auc = roc_auc_score(all_labels, all_probs, average="macro")


print("\nFinal Evaluation Metrics (MobileNetV3 Scratch)")
print("-----------------------------------------------")
print(f"Validation Loss : {best_val:.4f}")

print("\nMicro Metrics")
print(f"Precision : {precision_micro:.4f}")
print(f"Recall    : {recall_micro:.4f}")
print(f"F1 Score  : {f1_micro:.4f}")

print("\nMacro Metrics")
print(f"Precision : {precision_macro:.4f}")
print(f"Recall    : {recall_macro:.4f}")
print(f"F1 Score  : {f1_macro:.4f}")

print("\nROC-AUC Score :", roc_auc)


# ------------------------
# ROC Curve
# ------------------------

plt.figure(figsize=(8,6))

for i in range(all_labels.shape[1]):

    fpr, tpr, _ = roc_curve(all_labels[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (MobileNetV3 Scratch Multi-label)")
plt.legend()

plt.show()